# Notebook 05 — KPI Computation & Final Data Load

> Compute all KPIs, save output files for Tableau and reporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

matches = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')
print('Loaded.')

## KPI 1 — Team Win Rate

In [ ]:
valid = matches[matches['winner'].notna()]
team_matches = pd.concat([
    matches[['season','team1']].rename(columns={'team1':'team'}),
    matches[['season','team2']].rename(columns={'team2':'team'})
]).groupby('team').size().reset_index(name='total_matches')
team_wins = valid['winner'].value_counts().reset_index()
team_wins.columns = ['team','wins']
team_kpi = team_matches.merge(team_wins, on='team', how='left').fillna(0)
team_kpi['win_rate'] = (team_kpi['wins'] / team_kpi['total_matches'] * 100).round(2)
team_kpi = team_kpi.sort_values('win_rate', ascending=False)
team_kpi.to_csv('../data/processed/kpi_teams.csv', index=False)
team_kpi

## KPI 2 — Batter Strike Rate (min 200 balls)

In [ ]:
balls_faced = deliveries.groupby('batter').size().reset_index(name='balls')
runs_scored = deliveries.groupby('batter')['batsman_runs'].sum().reset_index()
bat_kpi = balls_faced.merge(runs_scored, on='batter')
bat_kpi['strike_rate'] = (bat_kpi['batsman_runs'] / bat_kpi['balls'] * 100).round(2)
bat_kpi = bat_kpi[bat_kpi['balls']>=200].sort_values('strike_rate', ascending=False)
bat_kpi.to_csv('../data/processed/kpi_batters.csv', index=False)
bat_kpi.head(10)

## KPI 3 — Bowler Economy Rate (min 200 balls)

In [ ]:
bowler_balls = deliveries.groupby('bowler').size().reset_index(name='balls')
bowler_runs = deliveries.groupby('bowler')['total_runs'].sum().reset_index()
bowler_kpi = bowler_balls.merge(bowler_runs, on='bowler')
bowler_kpi['economy'] = (bowler_kpi['total_runs'] / bowler_kpi['balls'] * 6).round(2)
bowler_wkts = deliveries[deliveries['is_wicket']==1].groupby('bowler').size().reset_index(name='wickets')
bowler_kpi = bowler_kpi.merge(bowler_wkts, on='bowler', how='left').fillna(0)
bowler_kpi = bowler_kpi[bowler_kpi['balls']>=200].sort_values('economy')
bowler_kpi.to_csv('../data/processed/kpi_bowlers.csv', index=False)
bowler_kpi.head(10)

## KPI 4 — Boundary % by Phase

In [ ]:
deliveries['is_boundary'] = deliveries['batsman_runs'].isin([4,6]).astype(int)
phase_boundary = deliveries.groupby('phase').agg(
    boundaries=('is_boundary','sum'),
    balls=('ball','count')
).reset_index()
phase_boundary['boundary_pct'] = (phase_boundary['boundaries'] / phase_boundary['balls'] * 100).round(2)
phase_boundary.to_csv('../data/processed/kpi_phases.csv', index=False)
phase_boundary

## KPI 5 — Toss Win Rate by Decision

In [ ]:
toss_kpi = valid.groupby('toss_decision')['toss_win_match'].mean() * 100
print('Toss Win Rate by Decision:')
print(toss_kpi.round(2))

## KPI 6 — Season Avg Scores & Run Differential

In [ ]:
match_scores = deliveries.groupby(['match_id','inning'])['total_runs'].sum().reset_index()
inn1 = match_scores[match_scores['inning']==1].rename(columns={'total_runs':'inn1'})
inn2 = match_scores[match_scores['inning']==2].rename(columns={'total_runs':'inn2'})
inn_both = inn1.merge(inn2, on='match_id').merge(matches[['id','season']], left_on='match_id', right_on='id')
inn_both['run_diff'] = inn_both['inn1'] - inn_both['inn2']
season_runs = inn_both.groupby('season')[['inn1','inn2','run_diff']].mean().round(1)
season_runs.to_csv('../data/processed/kpi_seasons.csv')
season_runs

## Final KPI Summary Visualisation

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('IPL Key Performance Indicators Dashboard', fontsize=14, fontweight='bold')
top8 = team_kpi[team_kpi['total_matches']>=50].head(8)
axes[0,0].barh(top8['team'], top8['win_rate'], color='#4e79a7')
axes[0,0].set_title('Team Win Rate (%)')
top_sr = bat_kpi.head(10)
axes[0,1].barh(top_sr['batter'], top_sr['strike_rate'], color='#f28e2b')
axes[0,1].set_title('Strike Rate — Top 10 Batters')
top_eco = bowler_kpi.head(10)
axes[1,0].barh(top_eco['bowler'], top_eco['economy'], color='#59a14f')
axes[1,0].set_title('Economy Rate — Top 10 Bowlers')
season_runs['inn1'].plot(ax=axes[1,1], marker='o', color='#e15759', label='1st Inn')
season_runs['inn2'].plot(ax=axes[1,1], marker='s', color='#76b7b2', label='2nd Inn')
axes[1,1].set_title('Avg Score by Season'); axes[1,1].legend()
plt.tight_layout(); plt.savefig('../reports/figures/fig12_kpi_summary.png', dpi=150); plt.show()
print('[KPI COMPLETE] All files saved to data/processed/')